# Group Model Comparison
### IT2011 — Final Evaluation: Progress Review II (Group Requirement, 10 marks)
**Group ID:** 2026-Y2-S1-[XX]
**Members:** IT25101779 · IT25101746 · IT25101789 · IT25101763 · IT25101786 · IT25101771

This notebook satisfies the group requirement: compare the optimum results from all 6
individually-trained models and discuss challenges, performance, and expected behavior. Each
member's full working is in their own notebook in this folder — this notebook summarizes and
compares the final results only.


## 1. Individual Model Summary| IT Number | Model | Category | Best Test Accuracy | Best Test F1 | Train-Test Gap ||---|---|---|---|---|---|| IT25101779 | Logistic Regression | Classification | 0.7552 | 0.76 (weighted) | 0.046 (4.6pp) || IT25101746 | Linear SVM | Classification | 0.7242 | 0.72 (weighted) | 0.018 (1.8pp) || IT25101789 | Random Forest (best of DT/RF) | Classification | 0.6595 | 0.69 (positive-class) / 0.66 (weighted) | 0.037 (3.7pp) || IT25101763 | K-Means (k=2) | Clustering | N/A -- see silhouette / ARI instead | N/A | silhouette: train 0.0034, val 0.0044 || IT25101786 | PCA + Logistic Regression (100 components) | Dimensionality Reduction | 0.6482 | N/A (not captured) | 0.024 (2.4pp) || IT25101771 | DistilBERT (2 epochs) | Deep Learning | **0.8377** | **0.84 (weighted)** | 0.048 (4.8pp) |**Train-Test Gap** is each member's own overfitting diagnostic from their notebook (trainaccuracy minus test accuracy). All six models show healthy gaps (under 5 percentage points),meaning none show strong evidence of overfitting -- IT25101779's Logistic Regression originallyshowed a 21.1-point gap before `min_df=5` and a lower `C=0.2` were applied, confirming the fixworked.**Note on Decision Tree / Random Forest's F1:** the 0.69 figure for Random Forest (and thediscarded Decision Tree's 0.67) reflects positive-class F1 only (`f1_score()` default). RandomForest's actual *weighted* F1 (which accounts for both classes fairly) is 0.66, and theDecision Tree's is only 0.49 -- its classification report shows a strong bias toward predicting"Positive" (0.90 recall) at the cost of "Negative" (0.20 recall). The Decision Tree istherefore not considered competitive and is excluded from the final-model discussion below infavor of its own Random Forest variant.**Note on K-Means:** both silhouette scores are very close to zero (0.0034 train, 0.0044validation), indicating the clusters found are barely more separated than random groupings ofthe data. This is itself an informative, reportable result (see Section 3b.1 discussion) --it is not a sign of a implementation error.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

comparison = pd.DataFrame({
    "Model": ["Logistic Regression\n(IT25101779)", "Linear SVM\n(IT25101746)",
              "Random Forest\n(IT25101789)", "PCA + LogReg\n(IT25101786)",
              "DistilBERT\n(IT25101771)"],
    "Accuracy": [0.7552, 0.7242, 0.6595, 0.6482, 0.8377],
    "F1": [0.76, 0.72, 0.66, None, 0.84],   # weighted F1; PCA's full classification report wasn't captured
})
comparison


**Note on K-Means (IT25101763):** clustering is unsupervised and is not directly comparable
to the supervised models' accuracy/F1, since it never sees the label during training. Its
result is reported separately using silhouette score and Adjusted Rand Index against the true
label (see IT25101763's notebook, Section 4) — a low ARI is still a valid and informative
finding, showing that raw feature similarity alone does not fully capture sentiment.


## 2. Comparison Chart

Once all `None` values above are filled in with real results, this chart visualizes the
supervised models side by side.


In [ ]:
plot_df = comparison.dropna()
plot_df_melted = plot_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df_melted, x="Model", y="Score", hue="Metric")
plt.ylim(0.6, 0.9)
plt.title("Test-Set Performance Across All Supervised Models")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


## 3. Discussion: Challenges, Performance & Expected Behavior**Actual performance ordering:** DistilBERT (83.8%) > Logistic Regression (75.5%) > Linear SVM(72.4%) > Random Forest (66.0%) > PCA + Logistic Regression (64.8%) > Decision Tree (55.0%,excluded as non-competitive) > K-Means (not comparable; silhouette ≈ 0). This matches thegroup's expected ordering almost exactly, confirming that context-aware deep learningoutperforms linear bag-of-words models, which in turn outperform tree ensembles and compressedrepresentations on this kind of sparse, high-dimensional text data.**Why DistilBERT leads.** As established in the group's Step 4 error analysis, TF-IDF-basedmodels structurally cannot resolve negation and contrastive conjunctions ("but", "however") theway a context-aware transformer can. This is a shared limitation across four of the six models(Logistic Regression, SVM, Random Forest, PCA), not a flaw unique to any one implementation --and it directly explains the roughly 8-point accuracy gap between DistilBERT and the next-bestmodel.**Why the Decision Tree underperformed so badly.** At 55.0% accuracy -- barely above the 50%majority-class baseline -- the Decision Tree is not a usable model on its own. Itsclassification report reveals why: 0.90 recall on "Positive" but only 0.20 recall on"Negative", meaning it learned to predict "Positive" most of the time rather than genuinelyseparating the two classes. The Random Forest variant corrects much of this bias (66.0%accuracy, more balanced recall) by averaging across many trees, which is exactly the ensemblingbenefit the group expected going into this comparison.**Challenges encountered:**- **Compute time:** DistilBERT's multi-epoch tuning took meaningfully longer than any classical  model, which constrained how many hyperparameter varieties could be explored within a  classroom compute budget.- **PCA memory constraints:** true (dense) PCA required capping the vocabulary to 1,000 features  to stay memory-feasible, unlike the other models which used a 4,000-10,000 feature TF-IDF  vocabulary -- a likely contributor to its comparatively lower 64.8% accuracy.- **Clustering ambiguity:** K-Means produced near-zero silhouette scores (0.0034 train, 0.0044  validation), indicating the data does not separate into natural, well-defined clusters under  raw TF-IDF similarity. This was a useful, concrete lesson in the difference between  unsupervised feature-similarity grouping and supervised sentiment classification -- they are  not the same objective, and a low silhouette score here is a valid finding, not a failed  experiment.- **An early overfitting catch:** IT25101779's first Logistic Regression run showed a 21.1-point  train-test gap, traced to `ngram_range=(1,2)` producing many rare bigrams the model could  memorize given only 8,530 training rows. Adding `min_df=5` and lowering `C` to 0.2 reduced the  gap to a healthy 4.6 points without hurting test accuracy -- a concrete example of diagnosing  and fixing overfitting rather than just reporting it.**Group conclusion:** DistilBERT (IT25101771) is the group's selected final model, on the basisof its clear accuracy lead (83.8% vs. the next-best 75.5%), healthy generalization (4.8-pointtrain-test gap), and balanced precision/recall across both sentiment classes. The trade-offacknowledged by the group is practical, not statistical: DistilBERT is slower to train and lessinterpretable than Logistic Regression, which remains a reasonable lightweight fallback wherecompute budget or explainability is a priority.

## 3b. Choosing the Final ModelThe assignment asks the group to choose one best model overall. We select it using **twocriteria together**, not accuracy alone:1. **Raw performance** -- test accuracy and F1 from Section 1's table.2. **Generalization** -- the Train-Test Gap from the same table. A model with top accuracy but   a large gap is a risk: it may be partly memorizing the training set, and could perform worse   than its test score suggests on genuinely new reviews.**Decision rule:** prefer the highest-accuracy model *unless* its gap is substantially largerthan the alternatives, in which case prefer the model with the best balance of the two.

In [ ]:
selection_df = comparison.copy()
selection_df["Train-Test Gap (pp)"] = [4.6, 1.8, 3.7, 2.4, 4.8]

print(selection_df)
print()
print("Final model selected: DistilBERT (IT25101771)")
print("Reason: highest test accuracy (83.8%) by a clear margin over every other model (next")
print("best is Logistic Regression at 75.5%), with a healthy train-test gap (4.8pp) showing")
print("no meaningful overfitting. Its architectural advantage -- reading text in context rather")
print("than as an unordered bag of words -- directly explains why it outperforms every TF-IDF-")
print("based model, consistent with the negation/contrastive-conjunction error patterns found")
print("in the group's earlier error analysis (Section 5.4 of the report).")


## 4. Individual Contribution Reference

| IT Number | Notebook | Model |
|---|---|---|
| IT25101779 | `IT25101779_Logistic_Regression.ipynb` | Logistic Regression |
| IT25101746 | `IT25101746_Linear_SVM.ipynb` | Linear SVM |
| IT25101789 | `IT25101789_Decision_Tree_Random_Forest.ipynb` | Decision Tree & Random Forest |
| IT25101763 | `IT25101763_KMeans_Clustering.ipynb` | K-Means Clustering |
| IT25101786 | `IT25101786_PCA_Dimensionality_Reduction.ipynb` | PCA |
| IT25101771 | `IT25101771_DistilBERT_Transformer.ipynb` | DistilBERT |
